## Preparo do ambiente

In [1]:
# Instalando bibliotecas
#!pip install torch_geometric
#!pip install setproctitle

In [ ]:
import sys
print(sys.executable)

In [2]:
# Imports
from abregrafo import bizoia_resultados
import pickle
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.sparse import coo_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay, auc, mean_squared_error
import xgboost as xgb

/home/luiz.gontijo/TCC_IC/OpenGraph/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Montando Drive
#from google.colab import drive
#drive.mount('/content/drive/')

### Convertendo SAML-D

In [4]:
!ls

Analise_SAML_D.ipynb  OpenGraph.ipynb  abregrafo.py	 link_prediction
History		      README.md        datasets		 node_classification
LICENSE		      Teste.ipynb      graph_generation  requirements.txt
Models		      __pycache__      imgs		 venv


In [6]:
# Leitura do arquivo
df = saml_d = pd.read_csv('../Datasets/SAML-D/SAML-D.csv')

In [7]:
# Convertendo Date e Time em Timestamp Unix
df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time']).astype('int64')

In [8]:
# Criando mapeamento numérico para labels em string

currencies          = df.groupby('Payment_currency')['Payment_currency'].count().sort_values(ascending=False).keys()
bank_locations      = df.groupby('Sender_bank_location')['Sender_bank_location'].count().sort_values(ascending=False).keys()
payment_types       = df.groupby('Payment_type')['Payment_type'].count().sort_values(ascending=False).keys()
laundering_types    = df.groupby('Laundering_type')['Laundering_type'].count().sort_values(ascending=False).keys()
accounts            = pd.concat([df['Sender_account'], df['Receiver_account']]).unique()
launderers          = pd.concat([
                                  df[df['Is_laundering'] == 1]['Sender_account'],
                                  df[df['Is_laundering'] == 1]['Receiver_account']
                                ]).unique()
is_launderer        = pd.Series(accounts).isin(launderers).astype(int)

account_map         = {account:i for i, account in enumerate(accounts)}
currency_map        = {currencies[i]:i for i in range(len(currencies))}
bank_location_map   = {bank_locations[i]:i for i in range(len(bank_locations))}
payment_type_map    = {payment_types[i]:i for i in range(len(payment_types))}
laundering_type_map = {laundering_types[i]:i for i in range(len(laundering_types))}

In [9]:
Accounts = pd.DataFrame({
    'Accounts': accounts,
    'Launderers': pd.Series(accounts).isin(launderers).astype(int)
})
Accounts

,Accounts,Launderers
0,8724731955,0
1,1491989064,0
2,287305149,0
3,5376652437,0
4,9614186178,0
...,...,...
855455,5750497909,0
855456,351917955,0
855457,7236544862,0
855458,1927425677,0


#### Conversão em PyTorch Data

In [9]:
# Definindo edge_index e edge_attr

df_edge_index = pd.concat([
                            df['Sender_account'].map(account_map),
                            df['Receiver_account'].map(account_map)
                          ], axis=1)

df_edge_attr = pd.concat([
                  df[['timestamp', 'Amount']],
                  df['Payment_currency'].map(currency_map),
                  df['Received_currency'].map(currency_map),
                  df['Sender_bank_location'].map(bank_location_map),
                  df['Receiver_bank_location'].map(bank_location_map),
                  df['Payment_type'].map(payment_type_map),
                  df['Laundering_type'].map(laundering_type_map)
                ], axis=1)

In [10]:
# Convertendo o dataset em Pytorch Data

data = Data(

    x= torch.ones((len(accounts),1)), # Label dos nós
    y= torch.tensor(df['Is_laundering'].values), # Label das arestas

    edge_index= torch.tensor(df_edge_index.values, dtype=torch.long),

    edge_attr= torch.tensor(df_edge_attr.values, dtype=torch.float)
)

data

Data(x=[855460, 1], edge_index=[9504852, 2], edge_attr=[9504852, 8], y=[9504852])

#### Conversão em COO Matrix

In [12]:
# Convertendo para matriz esparsa

row = df['Sender_account'].map(account_map)
col = df['Receiver_account'].map(account_map)

values = df['Amount'].values

num_nodes = Accounts.shape[0]

adj = coo_matrix((values, (row, col)), shape=(num_nodes, num_nodes))
adj

<COOrdinate sparse matrix of dtype 'float64'
	with 9504852 stored elements and shape (855460, 855460)>

In [13]:
# Features dos nós de dummy
feats = np.random.random(size=(num_nodes, 8))
feats

array([[8.73020561e-01, 1.33697511e-01, 5.56770348e-01, ...,
        7.14935019e-01, 4.01164057e-02, 3.32772065e-01],
       [7.08949838e-01, 1.45335438e-01, 1.89028934e-01, ...,
        8.62626595e-01, 5.54664512e-02, 9.44053252e-01],
       [1.22747466e-01, 8.89996446e-02, 6.42922552e-01, ...,
        3.89737967e-01, 8.56283773e-04, 8.77412316e-02],
       ...,
       [6.88945613e-01, 7.12312387e-01, 9.00678312e-01, ...,
        3.55086324e-01, 1.75474344e-01, 7.06181609e-01],
       [1.41733318e-01, 2.92746642e-01, 3.53079722e-01, ...,
        5.02686474e-01, 3.43501053e-01, 8.77003342e-01],
       [8.82333328e-01, 4.03805629e-01, 5.51328358e-01, ...,
        4.90546506e-01, 5.77976529e-01, 6.46577940e-01]],
      shape=(855460, 8))

In [14]:
labels = np.array(is_launderer)
#labels = 1 - is_launderer.values

In [15]:
# Configurando máscaras... não entendi essa parte ainda

train_mask = np.zeros(num_nodes, dtype=bool)
val_mask = np.zeros(num_nodes, dtype=bool)
test_mask = np.zeros(num_nodes, dtype=bool)

# Proporcões não podem se sobrepor!
div1 = num_nodes*0.3
div2 = num_nodes*0.3 + div1
train_mask[:int(div1)] = True
val_mask[int(div1):int(div2)] = True
test_mask[int(div2):] = True
#test_mask[:] = True

In [19]:
# Gravando pickles

pickle.dump(adj, open('datasets/saml_d/adj_-1.pkl', 'wb'))
pickle.dump(feats, open('datasets/saml_d/feats.pkl', 'wb'))
pickle.dump(labels, open('datasets/saml_d/label.pkl', 'wb'))
pickle.dump({
    'train': train_mask,
    'valid': val_mask,
    'test': test_mask
}, open('datasets/saml_d/mask_-1.pkl', 'wb'))

In [16]:
# Gravar apenas a máscara (testes)
pickle.dump({
    'train': train_mask,
    'valid': val_mask,
    'test': test_mask
}, open('datasets/saml_d/mask_-1.pkl', 'wb'))

In [20]:
!ls datasets/saml_d

adj_-1.pkl  feats.pkl  label.pkl  mask_-1.pkl


## Rodando o modelo

In [21]:
%cd node_classification

/home/luiz.gontijo/TCC_IC/OpenGraph/node_classification


In [23]:
!python main.py --load pretrn_gen2 --epoch 0 --tstdata saml_d

2026-05-22 12:40:23.612437: Start
Dataset: saml_d, Node num: 855460, Edge num: 887497
/home/luiz.gontijo/TCC_IC/OpenGraph/node_classification/data_handler.py:62: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.reshape(np.power(degree, -0.5), [-1])
/home/luiz.gontijo/TCC_IC/OpenGraph/node_classification/data_handler.py:115: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  asym_adj = t.sparse.FloatTensor(idxs, vals, shape)
/home/luiz.gontijo/TCC_IC/OpenGraph/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
2026-05-22 12:40:38.019937: Load Data
Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0
2026-05-22 12:40:38.152199: Model Prepared
2026-05-22 12:40:38.211993: Model 

In [24]:
%cd ..

/home/luiz.gontijo/TCC_IC/OpenGraph


In [26]:
bizoia_resultados(2, Accounts.shape[0])

--- Resultado 0 ---
Nós embeddados: 342184
Nós labelados: 342184
Porcentagem de nós embeddados: 0.4

preds
0    342184
Name: count, dtype: int64

labels
0    341524
1       660
Name: count, dtype: int64

Accuracy 0: 0.9980712131484816
Precision 0: 0.0
Recall 0: 0.0


--- Resultado 1 ---
Nós embeddados: 342184
Nós labelados: 342184
Porcentagem de nós embeddados: 0.4

preds
0    342184
Name: count, dtype: int64

labels
0    341524
1       660
Name: count, dtype: int64

Accuracy 1: 0.9980712131484816
Precision 1: 0.0
Recall 1: 0.0


--- Média dos resultados ---
Accuracy: 0.9980712131484816
Precision: 0.0
Recall: 0.0



### Lendo os resultados do modelo

In [27]:
!ls

Analise_SAML_D.ipynb  OpenGraph.ipynb  abregrafo.py	 link_prediction
History		      README.md        datasets		 node_classification
LICENSE		      Teste.ipynb      graph_generation  requirements.txt
Models		      __pycache__      imgs		 venv


In [28]:
with open('node_classification/Resultados/predicoes/predict0.pkl', 'rb') as f:
    data = pickle.load(f)

preds = data["preds"]
labels = data["labels"]
nodes = data['nodes']

In [29]:
resultado = pd.DataFrame.from_dict({'preds': data["preds"], 'labels': data["labels"]})

In [30]:
with open('node_classification/Resultados/embeddings/embedding0.pkl', 'rb') as f:
    embed = pickle.load(f)
embed

[tensor([[-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03],
         [-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03],
         [-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03],
         ...,
         [-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03],
         [-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03],
         [-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03]], device='cuda:0'),
 tensor([[-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03],
         [-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03],
         [-4.4426e+00,  2.5285e-02, -4.8085e-02,  ...,  8.2837e-03,
           1.6334e-04, -4.1897e-03],
         ...,
        

In [31]:
len(embed[1336][167])

1024

In [32]:
s = 0
for e in embed:
    s += len(e)

print('Nós embeddados:', s)
print('Nós labelados:', len(labels))
print('Porcentagem de nós embeddados:', s/Accounts.shape[0])

Nós embeddados: 342184
Nós labelados: 342184
Porcentagem de nós embeddados: 0.4


In [33]:
resultado

,preds,labels
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
...,...,...
342179,0,0
342180,0,0
342181,0,0
342182,0,0


In [34]:
accuracy = accuracy_score(resultado['labels'], resultado['preds'])
precision = precision_score(resultado['labels'], resultado['preds'], zero_division=0)
recall = recall_score(resultado['labels'], resultado['preds'])

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.9980712131484816
Precision: 0.0
Recall: 0.0


In [36]:
labels_df = pd.DataFrame(labels, columns=['labels'])
labels_df

,labels
0,0
1,0
2,0
3,0
4,0
...,...
342179,0
342180,0
342181,0
342182,0


In [37]:
resultado['preds'].value_counts()

preds
0    342184
Name: count, dtype: int64

In [38]:
resultado['labels'].value_counts()

labels
0    341524
1       660
Name: count, dtype: int64

In [39]:
comparison = (resultado['labels'] == labels_df['labels']).all()
print(f"All values match: {comparison}")

# Show detailed comparison
print("\nDetailed comparison:")
print((resultado['labels'] == labels_df['labels']).value_counts())

# Show any mismatches
mismatches = resultado[resultado['labels'] != labels_df['labels']]
if len(mismatches) > 0:
    print(f"\nMismatches found at indices:")
    print(mismatches)
else:
    print("\nNo mismatches found!")

All values match: True

Detailed comparison:
labels
True    342184
Name: count, dtype: int64

No mismatches found!


In [40]:
concat_emb = torch.cat([e.cpu() for e in embed], dim=0)
embed_df = pd.DataFrame(concat_emb.numpy())
embed_df

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419
1,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419
2,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419
3,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419
4,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342179,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419
342180,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419
342181,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419
342182,-4.442597,0.025285,-0.048085,0.034271,0.039082,-0.078898,0.04773,0.009969,-0.105928,0.086109,...,-0.009746,0.002897,0.003802,0.011674,0.032771,0.006643,-0.000982,0.008284,0.000163,-0.00419


In [41]:
with open('datasets/citeseer/feats.pkl', 'rb') as f:
    feats = pickle.load(f)
feats_df = pd.DataFrame(feats)
feats_df

,0,1,2,3,4,5,6,7,8,9,...,3693,3694,3695,3696,3697,3698,3699,3700,3701,3702
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3322,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3323,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3324,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3325,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.027778,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Testando outros Classificadores

Primeiro, é feita a divisão de treinamento. Depois, basta treinar um dos classificadores, e então, rodar as métricas.

In [39]:
# Divisão de treinamento
train_x, test_x, train_y, test_y = train_test_split(embed_df, labels_df['labels'], test_size=0.4, random_state=18, stratify=labels_df['labels'])
train_y = np.ravel(train_y)
test_y = np.ravel(test_y)

#### Random Forest

In [40]:
classificador_randomforest = RandomForestClassifier(n_estimators=200, random_state=18, max_depth=4, class_weight='balanced')
classificador_randomforest.fit(train_x, train_y)
pred_y = classificador_randomforest.predict(test_x)

#### XGBoost

In [41]:
# embed_df possui a embedding dos 325888 nós
# is_launderer possui as classificações (0 ou 1) para os nós fraudadores e não fraudadores
scale_pos_weight = (labels_df[labels_df['labels'] == 1].size/labels_df[labels_df['labels'] == 0].size)**(1/2)

In [43]:
classificador_xgb = xgb.XGBClassifier(objective='binary:logistic', random_state=18, scale_pos_weight=scale_pos_weight)
classificador_xgb.fit(train_x, train_y)
pred_y = classificador_xgb.predict(test_x)

### Métricas

In [45]:
accuracy = accuracy_score(test_y, pred_y)
precision = precision_score(test_y, pred_y)
recall = recall_score(test_y, pred_y)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.99807121878516
Precision: 0.0
Recall: 0.0


/home/luiz.gontijo/TCC_IC/OpenGraph/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# Playground
---
Testes temporários ou desorganizados

In [ ]:
!ls

datasets	  History  LICENSE	    Models		 README.md
graph_generation  imgs	   link_prediction  node_classification


### Abrindo pikles

In [ ]:
%cd ..

/content/drive/MyDrive/Colab Notebooks/OpenGraph


In [29]:
!ls

Analise_SAML_D.ipynb  OpenGraph.ipynb  abregrafo.py	 link_prediction
History		      README.md        datasets		 node_classification
LICENSE		      Teste.ipynb      graph_generation  requirements.txt
Models		      __pycache__      imgs		 venv


In [33]:
with open('datasets/citeseer/adj_-1.pkl', 'rb') as f:
    data = pickle.load(f)
print(data)

<COOrdinate sparse matrix of dtype 'int64'
	with 10344 stored elements and shape (3333, 3333)>
  Coords	Values
  (0, 628)	1
  (1, 158)	1
  (1, 486)	1
  (1, 1097)	1
  (1, 2919)	1
  (1, 2933)	1
  (2, 3285)	1
  (3, 1431)	1
  (3, 3219)	1
  (4, 467)	1
  (5, 648)	1
  (6, 1501)	1
  (7, 1833)	1
  (7, 2137)	1
  (8, 178)	1
  (8, 1033)	1
  (9, 1007)	1
  (10, 1670)	1
  (10, 2622)	1
  (11, 2034)	1
  (12, 113)	1
  (12, 557)	1
  (12, 677)	1
  (12, 794)	1
  (12, 839)	1
  :	:
  (3329, 607)	1
  (608, 3330)	1
  (3330, 608)	1
  (609, 3329)	1
  (3329, 609)	1
  (610, 3331)	1
  (3331, 610)	1
  (611, 3330)	1
  (3330, 611)	1
  (612, 3332)	1
  (3332, 612)	1
  (613, 3332)	1
  (3332, 613)	1
  (614, 3328)	1
  (3328, 614)	1
  (615, 3331)	1
  (3331, 615)	1
  (616, 3329)	1
  (3329, 616)	1
  (617, 3329)	1
  (3329, 617)	1
  (618, 3332)	1
  (3332, 618)	1
  (619, 3329)	1
  (3329, 619)	1


In [40]:
with open('datasets/pubmed/feats.pkl', 'rb') as f:
    data = pickle.load(f)
print(data)

[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.0553972  0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.01141881 0.00467923 ... 0.         0.         0.        ]
 [0.05307648 0.         0.         ... 0.         0.         0.        ]
 [0.         0.01447173 0.         ... 0.         0.         0.        ]]


In [ ]:
with open('datasets/saml_d/label.pkl', 'rb') as f:
    data = pickle.load(f)
print(data)

[0 0 0 ... 0 0 0]


In [ ]:
is_launderer = pd.DataFrame(data)
is_launderer

,0
0,0
1,0
2,0
3,0
4,0
...,...
325883,0
325884,0
325885,0
325886,0


In [ ]:
len(data)

1048575

In [ ]:
with open('datasets/saml_d/mask_-1.pkl', 'rb') as f:
    data = pickle.load(f)
print(data)

{'train': array([False, False, False, ..., False, False, False]), 'valid': array([False, False, False, ..., False, False, False]), 'test': array([ True,  True,  True, ...,  True,  True,  True])}


In [ ]:
for v in ['train', 'valid', 'test']:
  for b_1, b1, b5 in zip(data_n1[v], data_1[v], data_5[v]):
    if not (b_1 == b1 and b1 == b5):
      print(b_1, b1, b5)
  print()

True False True
True False False
True False False
True False False
True False False
True False True
True False True
True False False
True False False
True False False
True False False
True False False
True False False
True False False
True False False
True False False
True False False
True False True
True False False
True False False
True False True
True False True
True False True
True False True
True False False
True False False
True False False
True False False
True False False
True False False
True True False
True False False
True False False
True True False
True True False
True False True
True False False
True False False
True False False
True False True
True False False
True False True
True False False
True False False
True False True
True False False
True False False
True False False
True False False
True False True
True False False
True False False
True False False
True False False
True False False
True False True
True False True
True False False
True False True
True False False

In [ ]:
with open('graph_generation/gen_results/datasets/gen_data_ecommerce/embedding_dict.pkl', 'rb') as f:
    data = pickle.load(f)
data

#### Escrevendo Pikles

In [ ]:
# Armazenando pkl para leitura no OpenGraph

with open('datasets/saml_d/saml_d_sparse_graph.pkl', 'wb') as f:
    pickle.dump(A, f)

graph_data = {
    'adj': A,
    'edge_index': data.edge_index,
    'edge_attr': data.edge_attr,
    'y': data.y
}

with open('datasets/saml_d/saml_d_sparse_graph.pkl', 'wb') as f:
    pickle.dump(graph_data, f)

In [ ]:
!ls

data_handler.py  main.py  model.py  params.py  __pycache__  Resultados	Utils
